# 3. Running on a remote machine over SSH

With `cluster_type="ssh"`, a decorated function runs on a machine you can log
in to: a workstation, a lab server, a cloud instance, a cluster's interactive
node. There is no scheduler; the job starts at once.

You need three things:

- a host name and a user name that work with `ssh`;
- a private key that logs you in without a password prompt (the end of this
  tutorial shows how to set one up);
- on the remote machine, a Python of the **same minor version** as the one
  running this notebook.

This tutorial covers the first connection and host keys, what clustrix does
on the remote machine, the environment it builds there, exceptions, dividing
a loop into jobs, environment settings, and cleaning up.

**About the outputs.** They were produced against a single-node test cluster
running on the same computer as the notebook: an OpenSSH server on
`127.0.0.1`, port `2222`, with the user `clx`. Replace those with your own
host and user. The timings show what is slow and what is fast; they are not a
benchmark.

In [1]:
import logging
import os
import socket
import sys
import time

import clustrix
from clustrix import cluster, configure, get_config

logging.basicConfig(level=logging.WARNING, format="%(name)s: %(message)s")


def timed(label, function, *args):
    started = time.perf_counter()
    result = function(*args)
    print(f"{label:<24} {time.perf_counter() - started:6.1f} s")
    return result

## Configure

Everything about *where* goes into `configure()`.

In [2]:
# cluster-required: a host you can reach with ssh
configure(
    cluster_type="ssh",
    cluster_host="127.0.0.1",          # your host, e.g. "login.cluster.example.edu"
    cluster_port=2222,                 # leave out for the usual port 22
    username="clx",
    key_file="~/.ssh/id_ed25519",
    remote_work_dir="~/clustrix-tutorial",
    replicate_local_environment=False,
)

- `key_file` is the private key to log in with. A leading `~` means your home
  directory on this computer.
- `remote_work_dir` is a directory on the remote machine that clustrix may
  create and fill. Everything it puts there stays under this one directory.
  The default is `~/.clustrix/jobs`. On a cluster, choose a filesystem meant
  for job data (often called scratch) if your home directory has a small
  quota.
- `replicate_local_environment=False` is explained in the section on the
  environment below. It keeps this tutorial fast.

A function that reports where it is:

In [3]:
@cluster
def where_am_i():
    import getpass
    import os
    import socket
    import sys

    return {
        "host": socket.gethostname(),
        "user": getpass.getuser(),
        "pid": os.getpid(),
        "directory": os.getcwd(),
        "python": sys.version.split()[0],
    }

## The first connection is refused

An SSH server proves its identity with a *host key*. Your computer keeps the
keys of the hosts you have accepted in `~/.ssh/known_hosts`. Clustrix refuses
to connect to a host whose key is not in that file, because a machine that is
merely pretending to be your cluster would otherwise receive your code and
your data.

So on a computer that has never connected to this host, the first call fails.
The message contains the fingerprint of the key that was offered.

In [4]:
from clustrix.ssh_security import HostKeyVerificationError

try:
    where_am_i()
except HostKeyVerificationError as error:
    print(error)

Host key verification failed for '[127.0.0.1]:2222': this host is not in your known_hosts file(s), so clustrix refused the connection rather than risk a machine-in-the-middle attack.
  Offered key: ssh-ed25519 SHA256:t71ZfprzQXrLFRY47Bhv+RehzAB+NGmKLeKfi1cK//c

To fix this:
  1. If you recognize and trust this host, add its key with:
       ssh-keyscan -p 2222 127.0.0.1 >> ~/.ssh/known_hosts
     then retry.
  2. If you understand the risk and want clustrix to trust unknown host keys automatically (NOT recommended -- this is exactly the behavior that enables MITM attacks), set on ClusterConfig:
       ssh_host_key_policy="auto_add"



If you have already used `ssh` to log in to your cluster from this computer,
you will never see this: the key is in `known_hosts` already.

There are two correct ways to proceed. Both come down to the same act:
**you** compare the fingerprint with one you trust, and then record the key.

### Way 1: with `ssh` tools

Look up the fingerprint your cluster's administrators publish (usually on the
documentation page about logging in) and compare it with the one in the
message above. `ssh-keyscan` fetches the key and `ssh-keygen -lf` prints its
fingerprint in the same format:

In [5]:
!ssh-keyscan -p 2222 127.0.0.1 2>/dev/null | ssh-keygen -lf -

256 SHA256:t71ZfprzQXrLFRY47Bhv+RehzAB+NGmKLeKfi1cK//c [127.0.0.1]:2222 (ED25519)


When the fingerprint matches the published one, record the key. (Logging in
once with plain `ssh clx@host` does the same: it shows the fingerprint, asks,
and stores the key when you answer yes.)

In [6]:
!ssh-keyscan -p 2222 127.0.0.1 2>/dev/null >> ~/.ssh/known_hosts
!ssh-keygen -F "[127.0.0.1]:2222" -f ~/.ssh/known_hosts | head -1

# Host [127.0.0.1]:2222 found: line 1 


### Way 2: the notebook widget

The configuration widget's **Test connection** button shows the same
fingerprint with a **Trust this host** button next to it, which records
exactly the key that was shown. See [tutorial 6](06_notebook_widget.ipynb).

### What not to do by habit

`configure(ssh_host_key_policy="auto_add")` makes clustrix accept and record
whatever key a new host offers, without showing it to you. That is the
behaviour host keys exist to prevent. It is there for throwaway machines whose
keys change on every boot; do not use it for a cluster you send real work
to. Clustrix honours it only when you set it yourself, not when it comes from
a configuration file found in the working directory.

## The first job

With the host known, the same call goes through. `INFO` logging is on for
this call and the next so that the steps are visible.

In [7]:
logging.getLogger("clustrix").setLevel(logging.INFO)

report = timed("first call", where_am_i)
report

clustrix.executor_connections: Using SSH key from config-key


clustrix.decorator: where_am_i runs as one job.
where_am_i:
  no loop at the top level of the function to split


clustrix.executor_schedulers: Setting up enhanced two-venv environment with GPU detection


clustrix.utils: Detecting GPU capabilities on remote cluster...


clustrix.utils: Setting up two-venv environment...


clustrix.utils: Built virtualenvs at /home/clx/clustrix-tutorial/envs/py313_0ae49ae781c3


clustrix.utils: No GPUs detected, using standard VENV2 setup...


clustrix.executor_schedulers: Two-venv setup successful, using: /home/clx/clustrix-tutorial/envs/py313_0ae49ae781c3/venv1_serialization/bin/python


first call                 11.6 s


{'host': 'vm',
 'user': 'clx',
 'pid': 20569,
 'directory': '/home/clx/clustrix-tutorial/job_1795773793_8c14ca63',
 'python': '3.13.18'}

The function ran as user `clx`, in another process, in a directory of its own
under `remote_work_dir`. This notebook, for comparison:

In [8]:
print({"host": socket.gethostname(), "pid": os.getpid(), "directory": os.getcwd()})

{'host': 'vm', 'pid': 19961, 'directory': '/home/researcher/analysis'}


(The host name is the same only because the test cluster is on this computer.)

What clustrix did for that one call:

1. opened an SSH connection, after checking the host key;
2. serialized the function and its arguments and uploaded them to a new job
   directory, `~/clustrix-tutorial/job_<number>_<random>`;
3. built a Python environment under `~/clustrix-tutorial/envs/` (the line
   `Built virtualenvs at ...`);
4. started the job, waited for it, and downloaded the result;
5. verified the result file against a key known only to this job before
   reading it, then removed the job directory.

Step 3 is the slow one, and it happens once.

In [9]:
report = timed("second call", where_am_i)

clustrix.executor_connections: Using SSH key from config-key


clustrix.decorator: where_am_i runs as one job.
where_am_i:
  no loop at the top level of the function to split


clustrix.executor_schedulers: Setting up enhanced two-venv environment with GPU detection


clustrix.utils: Detecting GPU capabilities on remote cluster...


clustrix.utils: Setting up two-venv environment...


clustrix.utils: Reusing existing virtualenvs at /home/clx/clustrix-tutorial/envs/py313_0ae49ae781c3


clustrix.utils: No GPUs detected, using standard VENV2 setup...


clustrix.executor_schedulers: Two-venv setup successful, using: /home/clx/clustrix-tutorial/envs/py313_0ae49ae781c3/venv1_serialization/bin/python


second call                 2.9 s


In [10]:
logging.getLogger("clustrix").setLevel(logging.WARNING)

`Reusing existing virtualenvs`: the environment is found by a key computed
from its contents (Python version and the list of packages), so every later
job with the same requirements starts immediately, in this session and in
future ones.

## The environment on the remote machine

Your function needs its imports to work over there. Clustrix reads the
packages a job needs from the function itself and installs them in the remote
environment, at the versions you have here.

In [11]:
import numpy as np


@cluster
def column_means(table):
    import numpy

    return table.mean(axis=0), numpy.__version__


means, remote_numpy = timed("numpy job (new env)", column_means, np.arange(12.0).reshape(4, 3))
print("means:", means)
print("numpy there:", remote_numpy, "| numpy here:", np.__version__)

numpy job (new env)        13.2 s
means: [4.5 5.5 6.5]
numpy there: 2.5.5 | numpy here: 2.5.5


This job needed NumPy, which the earlier one did not, so a second environment
was built for it. The array went there and the result came back as ordinary
Python objects; anything that can be pickled can be an argument or a return
value.

`replicate_local_environment` chooses between two policies:

| value | what is installed remotely | when to use it |
|---|---|---|
| `True` (the default) | every package in your local environment, as listed by `pip freeze` | you want the remote environment to match yours exactly; the first job takes as long as installing all of it |
| `False` | only the packages the function was found to import | faster first job; right for most functions |

Detection looks at the function's code and the code it ships with it. An
import hidden from it (a plugin loaded by name, for example) will be missing
with `False`; either use `True` or name the package in
`configure(cluster_packages=[...])`.

If the cluster already has a conda environment with what you need, name it
with `@cluster(environment="name")` and the function runs in that instead.

### The Python version must match

Functions are sent as pickled code objects, and those are not portable between
Python minor versions. Before building an environment, clustrix therefore
looks on the remote machine for an interpreter of the same minor version as
the one running this notebook (it tries `python3.13`-style names as well as
`python3` and `python`) and uses that one. Above, the job reported Python
3.13.18 because this notebook runs 3.13.

If the remote machine has no matching interpreter, the job is not submitted.
The error (quoted from the source; it could not be provoked on the test
cluster, which has a matching Python) reads:

```text
RuntimeError: The remote system has Python 3.9, but this session runs Python
3.13. Serialized functions carry CPython bytecode, which cannot be loaded by a
different minor version, so the cluster needs a Python 3.13 interpreter -- or
conda, which clustrix will use to create one.
```

The fix is on one side or the other: run the notebook with a Python version
the cluster has, or make a matching one available there. On clusters that use
environment modules, `module_loads` (below) is how a newer Python is usually
put on the path.

If a matching interpreter exists but is not found under one of those names,
name it with `configure(python_executable="/opt/python/3.13/bin/python3")`.
Both remote environments are then built from it. It must exist on the remote
machine and have this session's minor version; a name that does not run
there is refused before any job is sent, with `python_executable='...' did
not run on the remote host`. Leave the setting at its default, `"python"`,
unless you need it. (This was tried on the test cluster with the path of its
Python 3.13, outside this notebook. On a host that has conda the setting does
not choose the interpreter, and that case was not run.)

## Exceptions

An exception raised on the remote machine is raised here, as itself, with the
remote traceback attached as its cause.

In [12]:
@cluster
def lookup(table, key):
    return table[key]


try:
    lookup({"alpha": 1}, "beta")
except KeyError as error:
    print("caught:", repr(error))
    print("cause: ", type(error.__cause__).__name__)
    print(error.__cause__)

caught: KeyError('beta')
cause:  RemoteTraceback

Traceback (most recent call last):
  File "<string>", line 55, in <module>
    result = func(*args, **kwargs)
  File "<dynamic-code>", line 3, in lookup
KeyError: 'beta'



Your function appears in the remote traceback under its own name, in
`<dynamic-code>`, because it travelled as code rather than as a file. When a
job fails, its directory is kept on the remote machine for inspection.

## A loop becomes several jobs

The loop division of tutorial 2 works here too. On a remote backend each
chunk of the loop is a job. `max_parallel_jobs` is the most jobs one call
will start (default 100); the loop is cut into at most that many chunks.

In [13]:
def hash_chain(seeds, length):
    import os

    finals, pids = {}, set()
    for seed in seeds:
        value = seed
        for step in range(length):
            value = (value * 6364136223846793005 + step) % 18446744073709551557
        finals[seed] = value
        pids.add(os.getpid())
    return finals, len(pids)


configure(max_parallel_jobs=4)

direct, _ = timed("direct, on this machine", hash_chain, list(range(8)), 20_000_000)
remote, processes = timed("over SSH, 4 jobs", cluster(hash_chain), list(range(8)), 20_000_000)

print("processes that did the work:", processes)
print("same answer:", remote == direct)

direct, on this machine    29.7 s


over SSH, 4 jobs           23.1 s
processes that did the work: 4
same answer: True


Each job costs a few seconds of overhead (upload, start, poll, download), so
this pays off only when the iterations are long. The 2-million-step loop of
tutorial 2 takes about three seconds directly, which is no more than the
overhead of one job (compare the second call above). Plain SSH has no scheduler:
the jobs start immediately and nothing limits them to the `cores`, `memory`
or `time` you name, so keep `max_parallel_jobs` at or below the number of
CPUs you may use on that machine. With SLURM (tutorial 4) the scheduler
enforces those.

## Environment variables, modules and setup commands

Three settings shape the shell a job runs in, in this order:

- `module_loads`: names passed to `module load`, for clusters that use
  environment modules (for example `["python/3.12", "gcc/13"]`);
- `environment_variables`: a dictionary of variables to export;
- `pre_execution_commands`: shell commands run just before the function.

In [14]:
configure(
    environment_variables={"STUDY_NAME": "pilot two"},
    pre_execution_commands=["export SCRATCH_ROOT=$(mktemp -d)"],
)


@cluster
def settings_seen():
    import os

    return os.environ.get("STUDY_NAME"), os.environ.get("SCRATCH_ROOT")


print(settings_seen())

configure(environment_variables=None, pre_execution_commands=None)

('pilot two', '/tmp/tmp.6xEZE3zEvv')


A module that cannot be loaded stops the job before it runs, with the
scheduler's own message. This test cluster has no module named `gcc`:

In [15]:
configure(module_loads=["gcc"])

try:
    settings_seen()
except RuntimeError as error:
    print(error)

configure(module_loads=None)

Could not look for Python on the remote host: ERROR: Unable to locate a modulefile for 'gcc'
clustrix: module load gcc failed, so the job was not run. Check module_loads.


## Where the files are, and cleaning up

After a successful job its directory is removed. Set
`cleanup_on_success=False` to keep it, which is the way to look at what a job
wrote. The `cluster_*` helpers list remote files through the same connection
settings.

In [16]:
from clustrix import cluster_ls

configure(cleanup_on_success=False)
kept = where_am_i()["directory"]
configure(cleanup_on_success=True)

print(kept)
for name in sorted(cluster_ls(kept, get_config())):
    print("   ", name)

/home/clx/clustrix-tutorial/job_1795773881_582ded9b
    .clustrix_result_key
    clustrix_env_setup.sh
    function_data.pkl
    function_deserialized.pkl
    job.err
    job.exit
    job.out
    job.sh
    result.pkl
    result.pkl.hmac
    result_raw.pkl


`function_data.pkl` is what was sent, `job.sh` the script that ran,
`result.pkl` what came back with its signature next to it, and `job.out` and
`job.err` hold what the job printed. That is where to look when a job
misbehaves.

The top of `remote_work_dir`:

In [17]:
work_dir = os.path.dirname(kept)          # the absolute form of ~/clustrix-tutorial
print(work_dir)
for name in sorted(cluster_ls(work_dir, get_config())):
    print("   ", name)

/home/clx/clustrix-tutorial


    envs
    job_1795773821_88f87f69
    job_1795773880_3f99a098
    job_1795773881_582ded9b


- `envs/` holds the environments. They are kept on purpose.
- `job_*` directories are those kept by `cleanup_on_success=False` or left by
  failed jobs.
- `data/` appears once input files have been staged (tutorial 5).

Clustrix never deletes `envs/` or `data/`. When you are done with a project,
remove the directory yourself. Staged input files are stored read-only, so
make the directory writable first:

```bash
ssh -p 2222 clx@127.0.0.1 'chmod -R u+w ~/clustrix-tutorial && rm -rf ~/clustrix-tutorial'
```

## Back to this machine

Set `cluster_host` to `None` and calls run here again. A decorated function
does not remember where it last ran.

In [18]:
configure(cluster_type="local", cluster_host=None)
print(where_am_i()["pid"] == os.getpid())

True


## Setting up a key

If `ssh you@host` asks for a password, create a key pair and install the
public half on the remote machine. In a terminal:

```bash
ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519      # choose a passphrase or leave it empty
ssh-copy-id -i ~/.ssh/id_ed25519.pub you@host   # asks for your password once
ssh you@host                                    # should now log in without a password
```

Then pass `key_file="~/.ssh/id_ed25519"` to `configure()`.

The notebook widget has a **Set up SSH key** button that performs the same
steps after asking for the password once (tutorial 6); the function behind it
is `clustrix.setup_ssh_keys(config, password)`.

If the remote machine only accepts passwords, do not put the password in your
code. Export it in the shell that starts Python and tell clustrix the name of
the variable: `configure(password_env_var="CLUSTER_PASSWORD")`. Clusters that
require a second factor on every login cannot be used non-interactively this
way; ask your administrators about key-based access.

## If something looks wrong

| You see | It means |
|---|---|
| `Host key verification failed for '...'` | The host is not in `~/.ssh/known_hosts`. Verify the fingerprint and record the key, as above. |
| Authentication failure | `username` or `key_file` is wrong, or the public key is not installed on the host. Check that `ssh -i <key> user@host` works. |
| `The remote system has Python X, but this session runs Python Y` | No interpreter of your local minor version was found remotely. Load a module that provides one, install one, name one with `python_executable`, or change the local version. |
| `module load ... failed, so the job was not run` | A name in `module_loads` does not exist on that cluster. `module avail` there lists the names. |
| `ModuleNotFoundError` in the remote traceback | A package the function needs was not detected. Use `replicate_local_environment=True` or `cluster_packages=[...]`. |
| The first call is slow, later ones are not | The environment was being built. It is reused from then on. |
| `FileNotFoundError` for a data file in the remote traceback | The file is on your computer, not on the cluster. See tutorial 5. |
| The function runs here although you configured SSH | `cluster_host` is empty, or `prefer_local_parallel` is set. |

## Next

[Tutorial 4](04_slurm.ipynb) submits the same functions to a SLURM scheduler.